# 🌐 Extracción Web Automatizada: Web Scraping y Consumo de APIs REST

### Autor: Néstor León | Business Intelligence & Analytics
---

Cuando queremos monitorizar a la competencia, seguir la evolución de precios inmobiliarios o recopilar estadísticas macroeconómicas, rara vez tenemos un botón de descarga directo. Aquí es donde el **Web Scraping** y el **consumo de APIs** se convierten en una habilidad imprescindible para cualquier analista de datos.

En este notebook vamos a trabajar dos vías de ingestión:
1. **Web Scraping ético y controlado con BeautifulSoup**: Parsearemos una página web simulando la extracción de tablas de precios y datos públicos, gestionando cabeceras `User-Agent` y tratando elementos dinámicos o anidados.
2. **Consumo de APIs REST con Python**: Realizaremos consultas HTTP a endpoints de datos públicos estructurados, parseando JSONs anidados y convirtiéndolos en DataFrames de `pandas` listos para analítica.


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import json
import time

print("Librerías importadas correctamente.")


Librerías importadas correctamente.


## 1. Web Scraping con BeautifulSoup: Extracción de Datos Tabulares

Vamos a simular un caso habitual: extraer los datos de una tabla HTML o de un listado de productos de una web. 

Para que los servidores no nos bloqueen la petición por parecer un bot automatizado, siempre debemos configurar una cabecera `User-Agent` realista que identifique a un navegador estándar.


In [2]:
# Definimos cabeceras para identificarnos como un navegador estándar
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Ejemplo reproducible: Estructura HTML simulada representativa de un listado inmobiliario/productos
html_sample = """
<html>
    <head><title>Portal Inmobiliario - Precios Valencia</title></head>
    <body>
        <div class="container">
            <h1>Oferta de Alquiler y Venta</h1>
            <div class="property-card" data-id="101">
                <h2 class="title">Piso en Ciutat Vella</h2>
                <span class="price">1250 €/mes</span>
                <span class="rooms">3 habs.</span>
                <span class="size">95 m2</span>
                <p class="neighborhood">El Carme</p>
            </div>
            <div class="property-card" data-id="102">
                <h2 class="title">Ático en Ruzafa</h2>
                <span class="price">1800 €/mes</span>
                <span class="rooms">2 habs.</span>
                <span class="size">85 m2</span>
                <p class="neighborhood">Eixample</p>
            </div>
            <div class="property-card" data-id="103">
                <h2 class="title">Estudio en Benimaclet</h2>
                <span class="price">750 €/mes</span>
                <span class="rooms">1 habs.</span>
                <span class="size">45 m2</span>
                <p class="neighborhood">Benimaclet</p>
            </div>
            <div class="property-card" data-id="104">
                <h2 class="title">Piso en El Pla del Real</h2>
                <span class="price">1400 €/mes</span>
                <span class="rooms">4 habs.</span>
                <span class="size">130 m2</span>
                <p class="neighborhood">Mestalla</p>
            </div>
        </div>
    </body>
</html>
"""

soup = BeautifulSoup(html_sample, 'html.parser')
print("Título extraído de la página:", soup.title.string)


Título extraído de la página: Portal Inmobiliario - Precios Valencia


### Extracción y Limpieza de Atributos

Recorremos cada tarjeta de inmueble (`div.property-card`), extraemos los campos de interés mediante selectores CSS y limpiamos las cadenas de texto (eliminando unidades como `€/mes` o `m2` para convertirlas a números operables).


In [3]:
properties = []
cards = soup.find_all('div', class_='property-card')

for card in cards:
    title = card.find('h2', class_='title').text.strip()
    price_raw = card.find('span', class_='price').text.strip()
    rooms_raw = card.find('span', class_='rooms').text.strip()
    size_raw = card.find('span', class_='size').text.strip()
    neighborhood = card.find('p', class_='neighborhood').text.strip()
    
    # Limpieza y conversión a tipos numéricos
    price = float(price_raw.replace('€/mes', '').replace('.', '').strip())
    rooms = int(rooms_raw.replace('habs.', '').strip())
    size = float(size_raw.replace('m2', '').strip())
    
    properties.append({
        'titulo': title,
        'barrio': neighborhood,
        'precio_eur': price,
        'habitaciones': rooms,
        'superficie_m2': size,
        'precio_m2': round(price / size, 2)
    })

df_scraping = pd.DataFrame(properties)
df_scraping


,titulo,barrio,precio_eur,habitaciones,superficie_m2,precio_m2
0,Piso en Ciutat Vella,El Carme,1250.0,3,95.0,13.16
1,Ático en Ruzafa,Eixample,1800.0,2,85.0,21.18
2,Estudio en Benimaclet,Benimaclet,750.0,1,45.0,16.67
3,Piso en El Pla del Real,Mestalla,1400.0,4,130.0,10.77


## 2. Consumo de APIs REST Públicas

Cuando una institución o servicio ofrece una API pública, siempre es preferible consumirla frente al scraping tradicional: es más estable, no rompe si cambia el diseño web y respeta los contratos de datos de la entidad.

A continuación implementamos un cliente con control de errores HTTP y parseo de respuestas JSON.


In [4]:
def fetch_api_data(url, params=None, timeout=10):
    """
    Función utilitaria robusta para consultar APIs REST con tratamiento de errores.
    """
    try:
        response = requests.get(url, params=params, headers=headers, timeout=timeout)
        response.raise_for_status() # Lanza excepción si el código es 4xx o 5xx
        return response.json()
    except requests.exceptions.HTTPError as err_h:
        print(f"Error HTTP: {err_h}")
    except requests.exceptions.ConnectionError as err_c:
        print(f"Error de Conexión: {err_c}")
    except requests.exceptions.Timeout as err_t:
        print(f"Timeout agotado: {err_t}")
    except Exception as err:
        print(f"Error inesperado: {err}")
    return None

# Simulamos la ingesta de un endpoint de datos macroeconómicos / demográficos abiertos
sample_api_response = {
    "indicador": "IPC General Mensual",
    "fuente": "Portal Estadístico",
    "unidad": "Indice 2021=100",
    "series": [
        {"periodo": "2023-01", "valor": 105.4, "tasa_anual": 5.9},
        {"periodo": "2023-02", "valor": 106.3, "tasa_anual": 6.0},
        {"periodo": "2023-03", "valor": 106.8, "tasa_anual": 3.3},
        {"periodo": "2023-04", "valor": 107.5, "tasa_anual": 4.1},
        {"periodo": "2023-05", "valor": 107.5, "tasa_anual": 3.2},
        {"periodo": "2023-06", "valor": 108.1, "tasa_anual": 1.9}
    ]
}

df_api = pd.DataFrame(sample_api_response["series"])
df_api["periodo"] = pd.to_datetime(df_api["periodo"])
df_api


,periodo,valor,tasa_anual
0,2023-01-01,105.4,5.9
1,2023-02-01,106.3,6.0
2,2023-03-01,106.8,3.3
3,2023-04-01,107.5,4.1
4,2023-05-01,107.5,3.2
5,2023-06-01,108.1,1.9


## 3. Conclusiones y Buenas Prácticas de Ingeniería de Datos

- **Resiliencia**: En producción, un scraper siempre debe incluir pausas de cortesía (`time.sleep`) entre peticiones para no saturar los servidores de destino.
- **Tipado**: Toda variable extraída en texto debe ser casteada de inmediato a enteros, flotantes o fechas válidas para evitar arrastrar errores a las capas de análisis.
- **Ventaja de APIs**: Siempre que existan endpoints oficiales (como los del INE o Banco de España), el pipeline debe priorizarlos frente al scraping de HTML.
